In [1]:
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv('amazon_reviews.csv')

In [3]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)

df.head()

Shape: (17340, 4)

Columns:
 Index(['sentiments', 'cleaned_review', 'cleaned_review_length',
       'review_score'],
      dtype='object')


,sentiments,cleaned_review,cleaned_review_length,review_score
0,positive,i wish would have gotten one earlier love it a...,19,5.0
1,neutral,i ve learned this lesson again open the packag...,88,1.0
2,neutral,it is so slow and lags find better option,9,2.0
3,neutral,roller ball stopped working within months of m...,12,1.0
4,neutral,i like the color and size but it few days out ...,21,1.0


In [4]:
df.rename(columns={
    'sentiments': 'sentiment',
    'cleaned_review': 'clean_text'
}, inplace=True)

In [10]:
print(df['clean_text'].isnull().sum())

3


In [11]:
df = df.dropna(subset=['clean_text'])

In [12]:
df.reset_index(drop=True, inplace=True)

In [13]:
def fix_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['review_score'].apply(fix_sentiment)

In [14]:
df['sentiment'].value_counts()

,count
sentiment,
Positive,11027
Negative,5193
Neutral,1117


In [15]:
X = df['clean_text']
y = df['sentiment']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [18]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [19]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

In [20]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

In [21]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

In [22]:
def evaluate(name, y_test, y_pred):
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("\n")

evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Naive Bayes", y_test, y_pred_nb)
evaluate("Decision Tree", y_test, y_pred_dt)

--- Logistic Regression ---
Accuracy: 0.6095732410611303
              precision    recall  f1-score   support

    Negative       0.43      0.20      0.27      1096
     Neutral       0.00      0.00      0.00       217
    Positive       0.64      0.88      0.74      2155

    accuracy                           0.61      3468
   macro avg       0.36      0.36      0.34      3468
weighted avg       0.53      0.61      0.55      3468



--- Naive Bayes ---
Accuracy: 0.6038062283737025
              precision    recall  f1-score   support

    Negative       0.45      0.45      0.45      1096
     Neutral       0.03      0.00      0.01       217
    Positive       0.68      0.74      0.71      2155

    accuracy                           0.60      3468
   macro avg       0.39      0.40      0.39      3468
weighted avg       0.57      0.60      0.59      3468



--- Decision Tree ---
Accuracy: 0.5135524798154556
              precision    recall  f1-score   support

    Negative       0.3